In [65]:
import cv2
import mediapipe as mp
import pandas as pd
import os
import warnings

# Ignore mediapipe warnings
warnings.filterwarnings("ignore")

# =======================================================
# SETTINGS: Change these variables before running the code
# =======================================================
target_letter = "Z"     # Letter to record (e.g., "A", "B", "C")
max_frames = 200        # Number of frames to record
# =======================================================

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

data = []
is_recording = False 

cap = cv2.VideoCapture(0)

print(f"\nInstructions for recording letter '{target_letter}':")
print(f"- Target: {max_frames} frames")
print("- Press 's' to start recording")
print("- Press 'q' to quit early and save the data")

while True:
    ret, frame = cap.read()
    if not ret: 
        break

    frame = cv2.flip(frame, 1) 
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb_frame)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            if is_recording:
                row = []
                h, w, c = frame.shape
                for lm in hand_landmarks.landmark:
                    row.extend([int(lm.x * w), int(lm.y * h)])
                
                row.append(target_letter)
                data.append(row)
                
                cv2.putText(frame, f"Recording: {target_letter} | Frames: {len(data)}/{max_frames}", 
                            (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                # Auto-stop when the target number of frames is reached
                if len(data) >= max_frames:
                    print(f"\nReached {max_frames} frames. Auto-saving...")
                    break
            else:
                cv2.putText(frame, f"Press 's' to record {target_letter}", 
                            (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Break the outer loop if we reached the max_frames
    if len(data) >= max_frames:
        break

    cv2.imshow("Data Collection", frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('s'): 
        is_recording = True
    elif key == ord('q'): 
        break

if cap.isOpened():
    cap.release()
cv2.destroyAllWindows()

# Process and save the collected data
if data:
    columns = []
    for i in range(21):
        columns.extend([f'x{i}', f'y{i}'])
    columns.append('label')
    
    df = pd.DataFrame(data, columns=columns)
    
    if os.path.exists('dataset.csv'):
        df.to_csv('dataset.csv', mode='a', header=False, index=False)
        print(f"Successfully appended {len(data)} samples for letter {target_letter} to dataset.csv!")
    else:
        df.to_csv('dataset.csv', index=False)
        print(f"Created dataset.csv and saved {len(data)} samples for letter {target_letter}!")
else:
    print("No data was recorded.")


Instructions for recording letter 'Z':
- Target: 200 frames
- Press 's' to start recording
- Press 'q' to quit early and save the data

Reached 200 frames. Auto-saving...
Successfully appended 200 samples for letter Z to dataset.csv!


In [66]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. قراءة البيانات
print("Loading data...")
df = pd.read_csv("dataset.csv")

# 2. فصل الميزات الإحداثيات (X) عن الحروف (y)
X = df.drop('label', axis=1)
y = df['label']

# 3. تقسيم البيانات للتدريب والاختبار (80% تدريب - 20% اختبار)
# استخدام stratify بيضمن إن نسبة الحروف A و B و C تكون متساوية في التدريب والاختبار
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. بناء الـ Pipeline (Scaling + Model)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 5. تدريب الموديل
print("Training the model...")
pipeline.fit(X_train, y_train)

# 6. تقييم الموديل
y_pred = pipeline.predict(X_test)
print("\n--- Model Evaluation ---")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 7. حفظ الموديل كملف واحد جاهز للتشغيل
with open("model.pkl", "wb") as f:
    pickle.dump(pipeline, f)
    
print("\nModel saved successfully as 'model.pkl'! Ready for Real-Time.")

Loading data...
Training the model...

--- Model Evaluation ---
Accuracy: 0.9971153846153846

Classification Report:
               precision    recall  f1-score   support

           A       1.00      1.00      1.00        40
           B       1.00      1.00      1.00        40
           C       1.00      1.00      1.00        40
           D       1.00      1.00      1.00        40
           E       1.00      1.00      1.00        40
           F       1.00      1.00      1.00        40
           G       1.00      1.00      1.00        40
           H       1.00      1.00      1.00        40
           I       0.97      0.98      0.98        60
           J       0.95      0.90      0.92        20
           K       1.00      1.00      1.00        40
           L       1.00      1.00      1.00        40
           M       1.00      1.00      1.00        40
           N       1.00      1.00      1.00        40
           O       1.00      1.00      1.00        40
           P     